In [1]:
import pandas as pd, json
C = "/home/ibm-user/shared/clean"
te = pd.read_pickle(f"{C}/te3.pkl")
TH = 0.99506049103294
hits = te[(te.laundering==1) & (te.score>=TH)].sort_values("tx_id").iloc[::400].head(3)
norm = te[(te.laundering==0) & (te.score<0.01)].sort_values("tx_id").iloc[::90000].head(2)
def row(r, tag):
    return {"txn_id": f"{tag}{r.tx_id}", "src_acct": str(r.src_acct), "dst_acct": str(r.dst_acct),
            "amount": float(r.amt_paid), "currency": r.cur_paid, "cur_recv": r.cur_recv,
            "src_bank": str(r.src_bank), "dst_bank": str(r.dst_bank), "fmt": r.fmt,
            "ts": r.ts.strftime("%Y-%m-%dT%H:%M:%S")}
demo = [row(r,"HOLD") for r in hits.itertuples()] + [row(r,"OK") for r in norm.itertuples()]
json.dump(demo, open("/home/ibm-user/shared/runtime_real/demo_txns.json","w"), indent=1)
print(len(demo), "transactions saved")

3 transactions saved


In [2]:
import requests, json, pandas as pd
C = "/home/ibm-user/shared/clean"
URL = "http://127.0.0.1:8001/screen"
d = pd.read_pickle(f"{C}/HI-Small.pkl")
demo = json.load(open("/home/ibm-user/shared/runtime_real/demo_txns.json"))

def to_payload(r, tag):
    return {"txn_id": f"{tag}{r.tx_id}", "src_acct": str(r.src_acct), "dst_acct": str(r.dst_acct),
            "amount": float(r.amt_paid), "currency": r.cur_paid, "cur_recv": r.cur_recv,
            "src_bank": str(r.src_bank), "dst_bank": str(r.dst_bank), "fmt": r.fmt,
            "ts": r.ts.strftime("%Y-%m-%dT%H:%M:%S")}

def warm(t):
    ts = pd.Timestamp(t["ts"]); lo = ts - pd.Timedelta("24h")
    s, dst = t["src_acct"], t["dst_acct"]
    prior = d[d.ts < ts]
    a = prior[(prior.src_acct == s) & (prior.ts >= lo)]
    last = prior[prior.src_acct == s].tail(1)                      # for the gap feature
    pair = prior[(prior.src_acct == s) & (prior.dst_acct == dst)].tail(1)   # for new_pair
    rc = prior[(prior.dst_acct == dst) & (prior.ts >= lo)].tail(300)        # receiver count
    h = pd.concat([last, pair, a, rc]).drop_duplicates("tx_id").sort_values("tx_id")
    for r in h.itertuples():
        requests.post(URL, json=to_payload(r, "W"))
    return len(h)

for t in demo:
    n = warm(t)
    r = requests.post(URL, json=t).json()
    print(t["txn_id"], "| warm-up rows:", n, "|", r["decision"], r["score"], "| audit_seq", r["audit_seq"])

HOLD4250810 | warm-up rows: 4 | release 0.000395 | audit_seq 9
OK4214445 | warm-up rows: 14 | release 0.0 | audit_seq 24
OK4307719 | warm-up rows: 3 | release 0.0 | audit_seq 28


In [4]:
%%writefile /home/ibm-user/shared/aml-harmcap/src/platform/deploy/Dockerfile.real
FROM jupyter-aml
WORKDIR /app
COPY audit/auditlog.py /app/audit/auditlog.py
COPY api/app_real.py api/scorer.py /app/api/
WORKDIR /app/api
ENV MODEL_PATH=/model/m3.txt RUNTIME_DIR=/data
EXPOSE 8000
CMD ["python", "-m", "uvicorn", "app_real:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting /home/ibm-user/shared/aml-harmcap/src/platform/deploy/Dockerfile.real


In [5]:
import requests, json, pandas as pd
C = "/home/ibm-user/shared/clean"
URL = "http://127.0.0.1:8002/screen"
d = pd.read_pickle(f"{C}/HI-Small.pkl")
demo = json.load(open("/home/ibm-user/shared/runtime_real/demo_txns.json"))

def to_payload(r, tag):
    return {"txn_id": f"{tag}{r.tx_id}", "src_acct": str(r.src_acct), "dst_acct": str(r.dst_acct),
            "amount": float(r.amt_paid), "currency": r.cur_paid, "cur_recv": r.cur_recv,
            "src_bank": str(r.src_bank), "dst_bank": str(r.dst_bank), "fmt": r.fmt,
            "ts": r.ts.strftime("%Y-%m-%dT%H:%M:%S")}

def warm(t):
    ts = pd.Timestamp(t["ts"]); lo = ts - pd.Timedelta("24h")
    s, dst = t["src_acct"], t["dst_acct"]
    prior = d[d.ts < ts]
    a = prior[(prior.src_acct == s) & (prior.ts >= lo)]
    last = prior[prior.src_acct == s].tail(1)
    pair = prior[(prior.src_acct == s) & (prior.dst_acct == dst)].tail(1)
    rc = prior[(prior.dst_acct == dst) & (prior.ts >= lo)].tail(300)
    h = pd.concat([last, pair, a, rc]).drop_duplicates("tx_id").sort_values("tx_id")
    for r in h.itertuples():
        requests.post(URL, json=to_payload(r, "W"))
    return len(h)

for t in demo:
    n = warm(t)
    r = requests.post(URL, json=t).json()
    print(t["txn_id"], "| warm-up rows:", n, "|", r["decision"], r["score"], "| audit_seq", r["audit_seq"])

HOLD4250810 | warm-up rows: 4 | release 0.09882 | audit_seq 5
OK4214445 | warm-up rows: 14 | release 0.0 | audit_seq 20
OK4307719 | warm-up rows: 3 | release 0.0 | audit_seq 24


In [6]:
import sys, math, numpy as np, pandas as pd
sys.path.insert(0, "/home/ibm-user/shared/aml-harmcap/src/platform/api")
from scorer import RealScorer, FMT
C = "/home/ibm-user/shared/clean"
d = pd.read_pickle(f"{C}/HI-Small.pkl")
te = pd.read_pickle(f"{C}/te3.pkl")
r0 = te[te.tx_id == 4250810].iloc[0]
cols = ["r24_s_cnt","r24_s_sum","r24_r_cnt","amt_ratio24","s_gap","new_pair"]
print("OFFLINE:", {c: r0[c] for c in cols}, "| score", round(r0.score, 4))

sec = lambda t: (t - pd.Timestamp("1970-01-01")).total_seconds()
s, dst, ts = r0.src_acct, r0.dst_acct, r0.ts
lo = ts - pd.Timedelta("24h")
prior = d[d.ts < ts]
a = prior[(prior.src_acct == s) & (prior.ts >= lo)]
last = prior[prior.src_acct == s].tail(1)
pair = prior[(prior.src_acct == s) & (prior.dst_acct == dst)].tail(1)
rc = prior[(prior.dst_acct == dst) & (prior.ts >= lo)].tail(300)
h = pd.concat([last, pair, a, rc]).drop_duplicates("tx_id").sort_values("tx_id")
SC2 = RealScorer(f"{C}/m3.txt")
for r in h.itertuples():
    SC2.update(r.src_acct, r.dst_acct, r.amt_paid, sec(r.ts))

ep = sec(ts)
sq = [(t, x) for t, x in SC2.sent[s] if t >= ep - 86400]
rq = [t for t in SC2.recv[dst] if t >= ep - 86400]
cnt, ssum = len(sq), sum(x for _, x in sq)
print("SERVING:", {"r24_s_cnt": cnt, "r24_s_sum": ssum, "r24_r_cnt": len(rq),
      "s_gap": (ep - SC2.last_send[s]) if s in SC2.last_send else None,
      "new_pair": 0 if (s, dst) in SC2.pairs else 1})

same_min_src = d[(d.src_acct == s) & (d.ts == ts) & (d.tx_id < r0.tx_id)]
same_min_dst = d[(d.dst_acct == dst) & (d.ts == ts) & (d.tx_id < r0.tx_id)]
print("earlier same-minute rows: from sender", len(same_min_src), "| to receiver", len(same_min_dst))

OFFLINE: {'r24_s_cnt': np.float64(0.0), 'r24_s_sum': np.float64(0.0), 'r24_r_cnt': np.float64(3.0), 'amt_ratio24': np.float64(nan), 's_gap': np.float64(0.0), 'new_pair': np.int64(1)} | score 1.0
SERVING: {'r24_s_cnt': 0, 'r24_s_sum': 0, 'r24_r_cnt': 3, 's_gap': 650640.0, 'new_pair': 1}
earlier same-minute rows: from sender 1 | to receiver 0
